In [6]:
import os
import urllib.request
import datetime
import pandas as pd
import re

In [7]:
def download_noaa_data(year1=1981, year2=2024):
    os.makedirs("data", exist_ok=True)
    for province_id in range(1, 28):
        existing_files = [f for f in os.listdir("data") if f.startswith(f"vhi_id_{province_id}_")]
        if existing_files:
            print(f"Дані для області {province_id} вже завантажені.")
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1={year1}&year2={year2}&type=Mean"
        try:
            req = urllib.request.urlopen(url)
            text = req.read().decode('utf-8')
            clean_text = re.sub(r'<[^>]+>', '', text)
            now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"data/vhi_id_{province_id}_{now}.csv"
            with open(filename, 'w') as f:
                f.write(clean_text)
            print(f"Завантажено: {filename}")
        except Exception as e:
            print(f"Помилка завантаження для ID {province_id}: {e}")

download_noaa_data()

Дані для області 1 вже завантажені.
Дані для області 2 вже завантажені.
Дані для області 3 вже завантажені.
Дані для області 4 вже завантажені.
Дані для області 5 вже завантажені.
Дані для області 6 вже завантажені.
Дані для області 7 вже завантажені.
Дані для області 8 вже завантажені.
Дані для області 9 вже завантажені.
Дані для області 10 вже завантажені.
Дані для області 11 вже завантажені.
Дані для області 12 вже завантажені.
Дані для області 13 вже завантажені.
Дані для області 14 вже завантажені.
Дані для області 15 вже завантажені.
Дані для області 16 вже завантажені.
Дані для області 17 вже завантажені.
Дані для області 18 вже завантажені.
Дані для області 19 вже завантажені.
Дані для області 20 вже завантажені.
Дані для області 21 вже завантажені.
Дані для області 22 вже завантажені.
Дані для області 23 вже завантажені.
Дані для області 24 вже завантажені.
Дані для області 25 вже завантажені.
Дані для області 26 вже завантажені.
Дані для області 27 вже завантажені.


In [8]:
def create_dataframe():
    files = [f for f in os.listdir("data") if f.endswith(".csv")]
    df_list = []
    for file in files:
        province_id = int(file.split("_")[2])
        filepath = os.path.join("data", file)
        df = pd.read_csv(filepath, index_col=False, header=1, skipinitialspace=True)
        df = df.dropna()
        df['province_id'] = province_id
        df_list.append(df)
        
    full_df = pd.concat(df_list, ignore_index=True)
    full_df = full_df[full_df['VHI'] >= 0]
    full_df.columns = full_df.columns.str.strip()
    return full_df

df = create_dataframe()
display(df.head())

,year,week,SMN,SMT,VCI,TCI,VHI,province_id
0,1982,1,0.059,258.24,51.11,48.78,49.95,10
1,1982,2,0.063,261.53,55.89,38.20,47.04,10
2,1982,3,0.063,263.45,57.30,32.69,44.99,10
3,1982,4,0.061,265.10,53.96,28.62,41.29,10
4,1982,5,0.058,266.42,46.87,28.57,37.72,10


In [9]:
dict_mapping = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

province_names = {
    1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
    6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
    11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
    16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
    21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'Республіка Крим',
    26: 'м. Київ', 27: 'м. Севастополь'
}

df['province_id'] = df['province_id'].map(dict_mapping)
df['province_name'] = df['province_id'].map(province_names)
display(df.head())

,year,week,SMN,SMT,VCI,TCI,VHI,province_id,province_name
0,1982,1,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
2,1982,3,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
3,1982,4,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
4,1982,5,0.058,266.42,46.87,28.57,37.72,21,Хмельницька


In [10]:
def get_vhi_for_year(df, province_id, year):
    return df[(df['province_id'] == province_id) & (df['year'] == year)][['week', 'VHI']]

def get_vhi_range(df, provinces, year_min, year_max):
    return df[(df['province_id'].isin(provinces)) & (df['year'] >= year_min) & (df['year'] <= year_max)]

def get_extremes(df, provinces, year):
    subset = df[(df['province_id'].isin(provinces)) & (df['year'] == year)]
    stats = {
        'Min VHI': subset['VHI'].min(),
        'Max VHI': subset['VHI'].max(),
        'Mean VHI': subset['VHI'].mean(),
        'Median VHI': subset['VHI'].median()
    }
    return pd.DataFrame([stats])

display(get_vhi_for_year(df, 1, 2010).head())
display(get_extremes(df, [9], 2020))

,week,VHI
34996,1,52.04
34997,2,51.05
34998,3,52.37
34999,4,52.69
35000,5,53.16


,Min VHI,Max VHI,Mean VHI,Median VHI
0,29.27,56.3,41.359423,39.375
